In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.retrieval import DenseRetriever, BM25Retriever, HybridRetriever, ColBERTRetriever
from src.evaluate import compute_retrieval_metrics

sns.set_theme(style='whitegrid')

In [ ]:
queries = []
with open('../data/msmarco_passage/queries.dev.tsv') as f:
    for line in f:
        qid, query = line.strip().split('\t', 1)
        queries.append((qid, query))

qrels = {}
with open('../data/msmarco_passage/qrels.dev.tsv') as f:
    for line in f:
        parts = line.strip().split()
        qid, pid, rel = parts[0], parts[2], int(parts[3])
        qrels.setdefault(qid, {})[pid] = rel

print(f'Queries: {len(queries)}  |  Qrels: {len(qrels)}')

In [ ]:
dense   = DenseRetriever()
bm25    = BM25Retriever(index_path='../indexes/msmarco_passage_bm25')
hybrid  = HybridRetriever(bm25, dense)
colbert = ColBERTRetriever(index_path='../indexes/msmarco_passage_colbert')

retrievers = {
    'BM25':    bm25,
    'Dense':   dense,
    'Hybrid':  hybrid,
    'ColBERT': colbert,
}

In [ ]:
TOP_K = 50
all_relevance = {}

for ret_name, retriever in retrievers.items():
    print(f'Running {ret_name} ...')
    relevance_lists = []
    for qid, query in queries:
        results  = retriever.retrieve(query, top_k=TOP_K)
        rel_list = [qrels.get(qid, {}).get(r['id'], 0) for r in results]
        relevance_lists.append(rel_list)
    all_relevance[ret_name] = relevance_lists
    print(f'  Done.')

In [ ]:
records = []
for ret_name, rel_lists in all_relevance.items():
    m = compute_retrieval_metrics(rel_lists, mrr_cutoffs=(5, 10, 20, 50), ndcg_cutoffs=(10, 100))
    m['Retriever'] = ret_name
    records.append(m)

df_results = pd.DataFrame(records).set_index('Retriever')
df_results.to_csv('../outputs/predictions/retrieval_metrics.csv')
print(df_results.round(4))

In [ ]:
mrr_cols = ['MRR@5', 'MRR@10', 'MRR@20', 'MRR@50']
cutoffs  = [5, 10, 20, 50]
colors   = {'BM25': '#E53935', 'Dense': '#1E88E5', 'Hybrid': '#43A047', 'ColBERT': '#FB8C00'}

plt.figure(figsize=(7, 4))
for ret_name in df_results.index:
    vals = df_results.loc[ret_name, mrr_cols].values.tolist()
    plt.plot(cutoffs, vals, marker='o', label=ret_name, color=colors[ret_name], linewidth=2)
plt.xlabel('Cutoff k')
plt.ylabel('MRR@k')
plt.title('MRR at Multiple Cutoffs')
plt.xticks(cutoffs)
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/retrieval_mrr_cutoffs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
metric_cols = ['nDCG@10', 'nDCG@100', 'AP']
x       = np.arange(len(metric_cols))
w       = 0.2
offsets = np.linspace(-1.5, 1.5, len(retrievers)) * w

plt.figure(figsize=(8, 4))
for i, ret_name in enumerate(df_results.index):
    vals = df_results.loc[ret_name, metric_cols].values
    plt.bar(x + offsets[i], vals, w, label=ret_name, color=colors[ret_name], edgecolor='white')
plt.xticks(x, metric_cols)
plt.ylabel('Score')
plt.title('nDCG and AP across Retrievers')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/retrieval_ndcg_ap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_heatmap = pd.DataFrame({'MS MARCO Passage': df_results['MRR@10'].to_dict()})

plt.figure(figsize=(6, 3))
sns.heatmap(df_heatmap, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, linecolor='white')
plt.title('MRR@10 across Retrievers and Datasets')
plt.tight_layout()
plt.savefig('../outputs/figures/retrieval_mrr10_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()